# Live Simulation: Slater's Rules (H → Ar)
Interactive notebook: first ionisation energy and orbital radius from Slater's effective nuclear charge.

**How to run:** `Run → Run All Cells` (Kernel → Restart Kernel and Run All Cells). Works in Jupyter, JupyterLab, JupyterLite (Pyodide) and Google Colab.

In [ ]:
%pip install -q numpy pandas matplotlib ipywidgets

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

## 1. Settings and reference data

In [ ]:
# ---- Adjustable parameters (same names as the report) ----
ONE_S_SCREENING            = 0.30   # electron in the 1s group
SAME_GROUP_SCREENING       = 0.35   # same (ns, np) group
PREVIOUS_SHELL_SCREENING   = 0.85   # shell n-1
INNER_SHELL_SCREENING      = 1.00   # shell n-2 and lower
RYDBERG_EV                 = 13.6057
BOHR_RADIUS_PM             = 52.9177
SCREENING_SCAN             = np.round(np.arange(0.20, 0.5001, 0.01), 2)   # 31 values

SYMBOLS = "H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar".split()

# NIST first ionisation energies (eV), H-Ar
REF_IE = np.array([13.5984, 24.5874, 5.3917, 9.3227, 8.2980, 11.2603, 14.5341, 13.6181, 17.4228,
                   21.5645, 5.1391, 7.6462, 5.9858, 8.1517, 10.4867, 10.3600, 12.9676, 15.7596])

# Clementi atomic radii (pm), Li-Ar
REF_R = {"Li":167,"Be":112,"B":87,"C":67,"N":56,"O":48,"F":42,"Ne":38,
         "Na":190,"Mg":145,"Al":118,"Si":111,"P":98,"S":88,"Cl":79,"Ar":71}

## 2. Model: Slater's Rules

In [ ]:
def configuration(Z):
    """Fill subshells 1s, 2s, 2p, 3s, 3p. Returns [(n, l, electrons), ...]."""
    left, out = Z, []
    for n, l, cap in [(1,0,2), (2,0,2), (2,1,6), (3,0,2), (3,1,6)]:
        if left <= 0: break
        e = min(cap, left); out.append((n, l, e)); left -= e
    return out

def z_eff(Z, cfg, target, k=SAME_GROUP_SCREENING):
    """Zeff = Z - S for one electron of subshell `target`."""
    nt = cfg[target][0]; S = 0.0
    for i, (n, l, e) in enumerate(cfg):
        if n == nt:                                   # same group
            same = (e - 1) if i == target else e
            S += same * (ONE_S_SCREENING if nt == 1 else k)
        elif n == nt - 1: S += e * PREVIOUS_SHELL_SCREENING
        elif n <= nt - 2: S += e * INNER_SHELL_SCREENING
    return Z - S

def energy(Z, cfg, k):
    """Energy proxy: sum of -13.6057 * Zeff^2 / n^2 over all electrons."""
    return sum(-e * RYDBERG_EV * z_eff(Z, cfg, t, k)**2 / n**2 for t, (n, l, e) in enumerate(cfg))

def properties(Z, k=SAME_GROUP_SCREENING):
    cfg = configuration(Z)
    ion = [list(x) for x in cfg]; ion[-1][2] -= 1               # remove one outer electron
    if ion[-1][2] == 0: ion.pop()
    ion = [tuple(x) for x in ion]
    t = len(cfg) - 1; n = cfg[t][0]; z = z_eff(Z, cfg, t, k)
    E_atom, E_ion = energy(Z, cfg, k), energy(Z, ion, k)
    return dict(cfg=cfg, n=n, zeff=z, S=Z - z, E_atom=E_atom, E_ion=E_ion,
                ie=E_ion - E_atom, radius=n**2 * BOHR_RADIUS_PM / z)

def cfg_text(cfg): return " ".join(f"{n}{'spdf'[l]}{e}" for n, l, e in cfg)

## 3. Results table (standard coefficient 0.35)

In [ ]:
rows = []
for Z in range(1, 19):
    p = properties(Z)
    rows.append({"Element": SYMBOLS[Z-1], "Configuration": cfg_text(p["cfg"]), "Outer Zeff": round(p["zeff"], 2),
                 "Atom energy (eV)": round(p["E_atom"], 2), "Ion energy (eV)": round(p["E_ion"], 2),
                 "IE model (eV)": round(p["ie"], 2), "IE reference (eV)": REF_IE[Z-1],
                 "Radius model (pm)": round(p["radius"], 1)})
table = pd.DataFrame(rows).set_index("Element")
table

## 4. Live simulation
Pick an element, change the same-group screening coefficient, or press ▶ (Run H→Ar) to step through all elements.

In [ ]:
ORBIT = lambda n: 0.4 * n + 0.2          # orbit radius in plot units

def draw_atom(Z, k, angle=0.0, ax=None):
    p = properties(Z, k); ax = ax or plt.gca()
    counts = {}
    for n, l, e in p["cfg"]: counts[n] = counts.get(n, 0) + e
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7)
    ax.add_patch(plt.Circle((0, 0), 0.14 + Z * 0.004, color="#c05a0b", zorder=3))
    ax.text(0, 0, f"{Z}+", ha="center", va="center", color="white", fontsize=9, weight="bold", zorder=4)
    for n, c in counts.items():
        r = ORBIT(n)
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color="#999", lw=1))
        for i in range(c):
            a = 2*np.pi*i/c + angle / n**1.3
            outer = (n == p["n"] and i == c - 1)
            ax.plot(r*np.cos(a), r*np.sin(a), "o", ms=9 if outer else 6,
                    color="#c05a0b" if outer else "#0f8b82", zorder=5)
    return p

def show(Z, k, angle):
    p = draw_atom(Z, k, angle, ax=ax_atom)   # reused figure axis below
    ref = REF_IE[Z-1]
    txt = (f"Element: {SYMBOLS[Z-1]} (Z = {Z})\nConfiguration: {cfg_text(p['cfg'])}\n"
           f"Shielding S: {p['S']:.2f}\nZeff (outer): {p['zeff']:.2f}\n"
           f"Ionisation energy: {p['ie']:.2f} eV\nReference (NIST): {ref:.2f} eV\n"
           f"Model error: {(p['ie']-ref)/ref*100:+.1f} %\nOrbital radius: {p['radius']:.1f} pm")
    ax_txt.clear(); ax_txt.axis("off"); ax_txt.text(0, .95, txt, va="top", fontsize=11, family="monospace")

fig_live, (ax_atom, ax_txt) = plt.subplots(1, 2, figsize=(9, 4.2), gridspec_kw={"width_ratios": [1, 1.1]})
plt.close(fig_live)

el   = w.Dropdown(options=[(f"{s}  (Z={i+1})", i+1) for i, s in enumerate(SYMBOLS)], value=6, description="Element")
ks   = w.FloatSlider(value=0.35, min=0.20, max=0.50, step=0.01, description="k (same group)", readout_format=".2f",
                     style={"description_width": "initial"}, layout=w.Layout(width="420px"))
ang  = w.FloatSlider(value=0, min=0, max=2*np.pi, step=0.05, description="angle", readout_format=".2f")
play = w.Play(value=0, min=0, max=10_000, step=1, interval=60, description="Spin")
auto = w.Play(value=1, min=1, max=18, step=1, interval=900, description="Run H→Ar")
out  = w.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        for a in (ax_atom,): a.clear()
        show(el.value, ks.value, ang.value)
        display(fig_live)

def spin(change): ang.value = (change["new"] * 0.08) % (2*np.pi)

def run_all(change): el.value = change["new"]

for wd in (el, ks, ang): wd.observe(refresh, names="value")
play.observe(spin, names="value"); auto.observe(run_all, names="value")

display(w.VBox([w.HBox([el, auto]), ks, w.HBox([ang, play]), out]))
refresh()

## 5. Ionisation energy and radius vs reference

In [ ]:
def validation(k=SAME_GROUP_SCREENING):
    P = [properties(Z, k) for Z in range(1, 19)]
    return np.array([p["ie"] for p in P]), np.array([p["radius"] for p in P])

ie_model, r_model = validation()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(SYMBOLS, ie_model, "o-", label="Slater model"); ax[0].plot(SYMBOLS, REF_IE, "s--", label="NIST reference")
ax[0].set(title="First ionisation energy", ylabel="eV"); ax[0].legend()
ax[1].plot(SYMBOLS, r_model, "o-", label="n²a₀ / Zeff")
ax[1].plot(SYMBOLS[2:], list(REF_R.values()), "s--", label="Clementi")
ax[1].set(title="Outer-orbital radius", ylabel="pm"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Statistics per period (MAE, MAPE, Pearson r)

In [ ]:
def stats(model, ref):
    model, ref = np.asarray(model), np.asarray(ref)
    return dict(MAE=np.mean(np.abs(model-ref)), MAPE=np.mean(np.abs((model-ref)/ref))*100,
                r=np.corrcoef(model, ref)[0, 1])

ref_r = np.array(list(REF_R.values()), float)          # Li..Ar
res = {}
for name, sl in [("Period 2", slice(2, 10)), ("Period 3", slice(10, 18))]:
    res[(name, "Ionisation energy")] = stats(ie_model[sl], REF_IE[sl])
    s2 = slice(sl.start - 2, sl.stop - 2)
    res[(name, "Radius")] = stats(r_model[sl], ref_r[s2])
pd.DataFrame(res).T.round(3)

## 7. Parameter study: error vs same-group coefficient

In [ ]:
mape_ie, mape_r = [], []
for k in SCREENING_SCAN:
    ie, r = validation(k)
    mape_ie.append(stats(ie[2:], REF_IE[2:])["MAPE"]); mape_r.append(stats(r[2:], ref_r)["MAPE"])
mape_ie, mape_r = np.array(mape_ie), np.array(mape_r)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, y, t in [(ax[0], mape_ie, "Ionisation-energy error"), (ax[1], mape_r, "Orbital-radius error")]:
    best = SCREENING_SCAN[y.argmin()]
    a.plot(SCREENING_SCAN, y, "o-"); a.axvline(best, ls=":", label=f"Best in scan: {best:.2f}")
    a.set(title=t, xlabel="Same-group screening coefficient", ylabel="MAPE (%)"); a.legend()
plt.tight_layout(); plt.show()